# Confronto tra Selection Sort e Quick Sort

**Laboratorio di Algoritmi e Strutture Dati — A.A. 2025/2026**

In questo notebook vengono implementati e confrontati gli algoritmi di ordinamento **Selection Sort** e **Quick Sort**.

## Obiettivo dell'esercizio

L'obiettivo dell'esercizio è confrontare **Selection Sort** e **Quick Sort** ed osservarne il comportamento al variare delle caratteristiche e della dimensione dell'input.

Il notebook contiene l'intero svolgimento dell'esercizio: implementazione degli algoritmi, verifica della correttezza, generazione dei dati di test, esecuzione degli esperimenti, presentazione dei risultati e loro analisi.

In accordo con la traccia:

- i dati di test vengono generati direttamente nel notebook;
- esecuzioni diverse del notebook generano dati diversi;
- i test vengono eseguiti nel notebook e producono direttamente i risultati;
- la documentazione del codice e la descrizione degli esperimenti sono riportate nelle celle Markdown.

Gli algoritmi di ordinamento sono implementati direttamente, senza utilizzare funzioni o librerie di ordinamento come loro sostituzione.

Il confronto sperimentale considera quattro tipologie di input:

1. valori distinti in ordine casuale;
2. valori già ordinati;
3. valori ordinati in senso inverso;
4. valori casuali con molti duplicati.

Le principali metriche considerate sono il **tempo di esecuzione**, il **numero di confronti** e il **numero di scambi**. Per il calcolo del numero di confronti e di scambi sono create due versioni strumentate a tale obiettivo dei due algoritmi, in modo tale che il conteggio non influisca sui tempi di esecuzione. Inoltre, il conteggio di confronti e scambi consente un'analisi indipendente dalle prestazioni dell'ambiente di esecuzione.

### Definizione delle operazioni conteggiate

Un **confronto** è un confronto tra valori dell'array effettuato per determinarne l'ordine.

Non vengono quindi conteggiati confronti relativi a indici, condizioni dei cicli o condizioni che controllano la ricorsione.

In particolare:

- in Selection Sort viene contato ogni confronto tra un elemento della parte non ordinata e il minimo corrente;
- in Quick Sort viene contato ogni confronto tra un elemento e il pivot effettuato durante la partizione.

Uno **scambio** corrisponde a ogni esecuzione dell'operazione che scambia due posizioni dell'array.

Vengono conteggiati anche gli **auto-scambi**, cioè i casi nei quali le due posizioni coincidono. Questa convenzione permette di contare direttamente le operazioni di scambio previste dall'algoritmo, senza introdurre condizioni aggiuntive nella versione strumentata.

Le versioni strumentate mantengono quindi lo stesso comportamento algoritmico delle versioni normali e aggiungono solamente l'aggiornamento dei contatori.

## Teoria e aspettative sperimentali

### Selection Sort

Selection Sort mantiene una parte iniziale dell'array già ordinata. A ogni iterazione individua il valore minimo nella parte ancora non ordinata e lo scambia con l'elemento che occupa la prima posizione di tale parte.

L'algoritmo opera **in-place** e, per individuare il minimo, deve esaminare progressivamente gli elementi della parte non ancora ordinata. Il numero di confronti non dipende quindi dall'ordinamento iniziale dei dati e il tempo di esecuzione è atteso di ordine quadratico, $\Theta(n^2)$, sia su input favorevoli sia su input sfavorevoli.

### Quick Sort

Quick Sort utilizza l'approccio **divide et impera**. L'array viene suddiviso rispetto a un pivot e i due sottoarray ottenuti vengono successivamente ordinati mediante chiamate ricorsive.

Nella versione utilizzata in questo notebook l'algoritmo è **ricorsivo e in-place**, utilizza come pivot l'ultimo elemento del sottoarray e adotta la **partizione come vista a lezione**, nella quale gli elementi minori o uguali al pivot vengono collocati nella parte sinistra.

Come visto a lezione, le prestazioni di Quick Sort dipendono dalla qualità delle partizioni ottenute. Quando le partizioni sono bilanciate il comportamento è dell'ordine di $\Theta(n \log n)$; quando invece risultano fortemente sbilanciate si può raggiungere il caso peggiore $\Theta(n^2)$. Con la scelta dell'ultimo elemento come pivot, un input già ordinato costituisce un caso particolarmente sfavorevole.

### Obiettivo del confronto

Gli esperimenti sservono a verificare quanto queste differenze teoriche siano osservabili nell'esecuzione reale. In particolare verrà studiato l'effetto della dimensione dell'input, dell'ordinamento iniziale e della presenza di valori duplicati sui tempi di esecuzione e sul numero di operazioni effettuate dai due algoritmi.

## Import e configurazione iniziale

In [1]:
# Le importazioni necessarie per generazione dei dati,
# benchmark e grafici saranno aggiunte nelle sezioni successive.

## Implementazione di Selection Sort

Viene inizialmente implementata la versione normale di **Selection Sort**, che sarà successivamente utilizzata per il benchmark temporale.

L'implementazione segue direttamente l'algoritmo visto nel corso: a ogni iterazione viene cercato il valore minimo nella parte dell'array non ancora ordinata e tale valore viene scambiato con quello nella posizione corrente.

L'algoritmo opera **in-place**, quindi modifica direttamente l'array ricevuto in ingresso senza creare un secondo array ordinato.

In [2]:
def selection_sort(A):
    """
    Ordina l'array A in ordine non decrescente utilizzando Selection Sort.

    L'algoritmo opera in-place: l'array ricevuto viene modificato
    direttamente e non viene restituito un nuovo array.

    Parametri:
    - A : list, Array di valori confrontabili da ordinare.

    Returns:
    - None, L'ordinamento viene effettuato direttamente su A.
    """

    n = len(A)

    # Dopo ogni iterazione, A[0:i+1] contiene gli elementi più piccoli dell'array nelle loro posizioni definitive
    for i in range(n - 1):
        min_index = i

        # Cerca il minimo nella parte dell'array non ancora ordinata
        for j in range(i + 1, n):
            if A[j] < A[min_index]:
                min_index = j

        # Porta il minimo trovato nella posizione corrente (anche quando min_index == i)
        A[i], A[min_index] = A[min_index], A[i]

### Input, output e funzionamento

La funzione `selection_sort(A)` riceve in ingresso una lista `A` contenente valori confrontabili e la ordina in ordine non decrescente. Operando **in-place**, il valore di ritorno è `None`

Sia `n` la lunghezza dell'array. L'algoritmo considera una posizione `i` alla volta, dalla prima fino alla penultima. Inizialmente assume che l'elemento in posizione `i` sia il minimo della parte dell'array ancora da ordinare. Attraversando le posizioni successive con l'indice `j`, aggiorna `min_index` ogni volta che incontra un valore più piccolo.

Terminata la ricerca, l'elemento minimo individuato viene scambiato con quello nella posizione `i`. Dopo lo scambio, l'elemento in posizione `i` si trova nella sua posizione definitiva.

Il procedimento viene eseguito soltanto per le prime `n-1` posizioni: quando queste sono state sistemate, anche l'ultimo elemento è necessariamente nella posizione corretta.

La ricerca del minimo viene effettuata per ogni posizione indipendentemente dall'ordinamento iniziale dell'input. Il numero di confronti tra valori è quindi

\[
(n-1) + (n-2) + \dots + 1
= \frac{n(n-1)}{2},
\]

e il tempo di esecuzione è $\Theta(n^2)$ sia nel caso migliore sia nel caso peggiore.